# Boundary-Preserving Organization: 4-State Lumpable vs Perturbed Demo

**Paper:** Vick, *Boundary-Preserving Organization in Dynamical Systems* (2025).  
**Purpose:** Minimal runnable example for adoption: closure energy \(E^{\mathrm{op}}_{\mathrm{cl}}(q)\), spectral gap, and greedy coarse-graining on a 4-state lumpable kernel vs a perturbed (non-lumpable) kernel.

In [ ]:
import sys
sys.path.insert(0, "../src")
sys.path.insert(0, "..")
import numpy as np
from boundary_org import (
    closure_energy,
    spectral_gap_abs,
    greedy_coarse_graining,
    graph_modularity_q,
    make_lumpable_block_diagonal,
    make_non_lumpable_perturbed,
)

## 1. Lumpable kernel (block-diagonal)

Partition \(q = \{B_1, B_2\}\) with \(B_1 = \{0,1\}, B_2 = \{2,3\}\). By construction \(K\) has no cross-block mass, so \(E^{\mathrm{op}}_{\mathrm{cl}}(q) = 0\) (Def 2.3, T3.1).

In [ ]:
rng = np.random.default_rng(42)
syn_lump = make_lumpable_block_diagonal(4, partition=[[0, 1], [2, 3]], rng=rng)
print("Lumpable: partition", syn_lump.partition)
E_lump = closure_energy(syn_lump.K, syn_lump.mu, syn_lump.partition)
print(f"Closure energy E_cl(q) = {E_lump:.2e}")
res = spectral_gap_abs(syn_lump.K, syn_lump.mu)
print(f"Spectral gap_abs = {res.gap_abs:.4f}, lambda_2 = {res.lambda_2:.4f}")
Q_lump = graph_modularity_q(syn_lump.K, syn_lump.partition)
print(f"Modularity Q = {Q_lump:.4f}")

## 2. Perturbed (non-lumpable) kernel

\(K = (1-\varepsilon) K_{\mathrm{lump}} + \varepsilon K_{\mathrm{noise}}\). Cross-block mass implies \(E^{\mathrm{op}}_{\mathrm{cl}}(q) > 0\).

In [ ]:
syn_pert = make_non_lumpable_perturbed(syn_lump, epsilon=0.1, rng=rng)
E_pert = closure_energy(syn_pert.K, syn_pert.mu, syn_pert.partition)
print(f"Perturbed: E_cl(q) = {E_pert:.4f}")
res_pert = spectral_gap_abs(syn_pert.K, syn_pert.mu)
print(f"gap_abs = {res_pert.gap_abs:.4f}")
Q_pert = graph_modularity_q(syn_pert.K, syn_pert.partition)
print(f"Modularity Q = {Q_pert:.4f}")

## 3. Greedy coarse-graining (T3.4)

From identity partition, merge the block pair that **minimally increases** \(E^{\mathrm{op}}_{\mathrm{cl}}\). Terminates in at most \(n-1\) steps.

In [ ]:
q_star, trajectory, steps = greedy_coarse_graining(syn_pert.K, syn_pert.mu)
print(f"Greedy: steps = {steps}, final partition = {q_star}")
print("Closure energy trajectory:", [round(e, 4) for e in trajectory])